# Processing ISFS High-Rate Data

This notebook processes M2HATS ISFS high-rate (geo-rotated, NOT tilt-corrected) NetCDF files from the surface network into a consolidated, analysis-ready Zarr store. It works in tandem with `m2hats_to_zarr.py`, which defines `restructure_m2hats`, the preprocessing helpers (`fix_time_and_t1`), and the variable metadata used in the final output.

**Workflow:**
1. **Setup** — Python imports, path configuration, and Dask cluster on Casper
2. **Fix a dimension error** — One source file has transposed `sample` dimensions; correct it before batch processing
3. **Build the file list** — Assemble the sorted source-file list, substituting the corrected file
4. **Write the combined Zarr store** — Concatenate all files day-by-day into a single Zarr store, with timestamp correction and instrument-swap padding (resume-safe)
5. **Validate the combined store** — Spot-check structure and values before restructuring
6. **Restructure into a DataTree** — Reorganize from a flat variable-per-site layout into a hierarchical DataTree with instrument groups and site/height coordinates
7. **Verify the output** — Inspect the store and compare a time slice against the raw NetCDF source

**Input:** `surface_highrate_geo/*.nc`  
**Output:** `GDEX_datasets/isfs_m2hats_qc_geo.zarr`

## Stage 1: Setup

### Imports

In [ ]:
import re
import importlib
from itertools import groupby
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import zarr as zarr_lib

import m2hats_to_zarr as m
from m2hats_to_zarr import restructure_m2hats, VAR_METADATA, fix_time_and_t1

### Configure paths

All data paths are set here. Update these if source files or output locations change.

In [ ]:
# Source high-rate NetCDF files
isfs_highrate_path = "/lustre/desc1/scratch/myasears/M2HATS/FDA_datasets/surface_highrate_geo"

# Intermediate combined Zarr store (Stage 4 output / Stages 5-6 input)
working_zarr = "/lustre/desc1/scratch/myasears/M2HATS/WORKING_datasets/isfs_m2hats_qc_geo_combined.zarr"

# Final restructured Zarr store (Stage 6 output)
gdex_zarr = "/lustre/desc1/scratch/myasears/M2HATS/GDEX_datasets/isfs_m2hats_qc_geo.zarr"

### Spin up a Dask cluster on Casper

Scale to 20 workers before running any computation. All file reads and Zarr writes are lazy Dask graphs, so the cluster must be running before any `.compute()` or `.to_zarr()` calls below.

In [ ]:
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

lustre_scratch = "/lustre/desc1/scratch/myasears"

cluster = PBSCluster(
    job_name = 'dask-eol-26',
    cores = 1,
    memory = '20GiB',
    processes = 1,
    local_directory = f"{lustre_scratch}/dask/spill/",
    log_directory = f"{lustre_scratch}/dask/logs/",
    resource_spec = 'select=1:ncpus=1:mem=8GB',
    queue = 'casper',
    walltime = '5:00:00',
    interface = 'ext'
)

client = Client(cluster)

cluster.scale(jobs=20)
client.wait_for_workers(20)

In [ ]:
cluster

## Stage 2: Correct a dimension error in one source file

One file — `isfs_m2hats_qc_geo_hr_20230728_06.nc` — has its `sample` (60) and `sample_60` (50) dimensions transposed relative to every other file in the dataset. The cells below inspect the mismatch against a reference file, rename the dimensions to the correct layout, and write a corrected copy to disk. The original errored file is excluded from the file list in Stage 3.

> **Note on time encoding:** All files store `time` as seconds within the current hour, measured from midnight of the file's day. The `base_time` variable holds the absolute hour offset. `fix_time_and_t1` (from `m2hats_to_zarr`) reconstructs correct absolute timestamps before concatenation.

### Inspect the dimension mismatch

In [ ]:
normal_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_03.nc"
normal_dataset = xr.open_dataset(normal_path)
print(normal_dataset.dims)

errored_path = f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06.nc"
errored_dataset = xr.open_dataset(errored_path)
print(errored_dataset.dims)

In [ ]:
errored_dataset = errored_dataset.rename({"sample": "__tmp_sample_50"})
errored_dataset = errored_dataset.rename({"sample_60": "sample"})
errored_dataset = errored_dataset.rename({"__tmp_sample_50": "sample_50"})

errored_dataset.to_netcdf(f"{isfs_highrate_path}/isfs_m2hats_qc_geo_hr_20230728_06_aligned.nc")

## Stage 3: Build the full file list

Glob all `.nc` files from the source directory in sorted order, then exclude the original errored file. The corrected aligned version written in Stage 2 is already present in the directory and will be picked up automatically.

In [ ]:
isfs_highrate_files = [
    f for f in sorted(Path(isfs_highrate_path).glob("*.nc"))
    if f.name != "isfs_m2hats_qc_geo_hr_20230728_06.nc"
]

## Stage 4: Write the combined Zarr store

Process files one day at a time: open each day's hourly files with `xr.open_mfdataset` (using Dask for parallel reads), apply preprocessing, and append the result to the Zarr store. The loop checks what has already been written on startup so it can be safely re-run to resume after an interruption.

Each file is preprocessed by `fix_time_and_t1` (from `m2hats_to_zarr`), which reconstructs absolute timestamps and pads NaN placeholders for the t1 sonic swap (CSAT3 → CSAT3A on 2023-08-23T13:00Z) so both instrument periods concatenate cleanly.

In [ ]:
def date_key(p):
    m = re.search(r'_(\d{8})_', p.name)
    return m.group(1) if m else "unknown"

# Detect which dates are already written so we can resume
try:
    existing = xr.open_zarr(working_zarr)
    last_written = str(existing.time.values[-1])[:10].replace("-", "")
    print(f"Resuming after {last_written}")
    existing.close()
    first_run = False
except (FileNotFoundError, zarr_lib.errors.GroupNotFoundError):
    last_written = None
    first_run = True

for date, group in groupby(isfs_highrate_files, key=date_key):
    day_files = list(group)
    if last_written and date <= last_written:
        print(f"Skipping {date} (already written)")
        continue

    print(f"Processing {date} ({len(day_files)} files)…")
    ds_day = xr.open_mfdataset(
        day_files,
        preprocess=fix_time_and_t1,
        combine="nested",
        concat_dim="time",
        parallel=True,
        chunks={"time": 3600},
        decode_timedelta=False,
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )

    ds_day = ds_day.chunk({"time": 3600})   # normalize NaN placeholders and all other vars

    if first_run:
        ds_day.to_zarr(working_zarr, mode="w", consolidated=True)
        first_run = False
    else:
        ds_day.to_zarr(working_zarr, append_dim="time")

    ds_day.close()
    print(f"  Done {date}")

## Stage 5: Validate the combined store

Reload the combined Zarr store and run a structural check before restructuring. The dry-run call to `restructure_m2hats` builds the DataTree in memory without writing to disk, making it fast to verify that dimension sizes, group layout, and data values all look correct.

In [ ]:
ds = xr.open_zarr(working_zarr)

### Preview the DataTree structure

Run `restructure_m2hats` in dry-run mode (`write=False`) on a small one-hour subset to confirm the expected group hierarchy and dimension sizes without triggering a full computation.

In [ ]:
subset = ds.isel(time=slice(0, 300)).persist()    # one hour, in cluster memory

tree_test = restructure_m2hats(
    subset,
    output_path="./m2hats_test.zarr",
    tilt_corrected=False,
    time_chunk_seconds=300,
    write=False,
)

for path in tree_test.groups:
    node = tree_test[path].ds
    if not node.data_vars:
        continue
    print(f"{path:35s} dims={dict(node.sizes)}")

Expected output on a pre-Aug-23 hour (or any subset that doesn't span the t1 swap):

```
/array/sonic_60hz           dims={'site': 22, 'time': 3600, 'sample': 60}
/array/sonic_50hz           dims={'site': 9,  'time': 3600, 'sample_50': 50}
/array/sonic_30hz           dims={'site': 21, 'time': 3600, 'sample_30': 30}
/array/irga_60hz            dims={'site': 16, 'time': 3600, 'sample': 60}
/array/barometer_20hz       dims={'site': 16, 'time': 3600, 'sample_20': 20}
/array/trh_1hz              dims={'site': 16, 'time': 3600}
/profile_t0/sonic_60hz      dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/irga_60hz       dims={'height': 8, 'time': 3600, 'sample': 60}
/profile_t0/barometer_20hz  dims={'height': 8, 'time': 3600, 'sample_20': 20}
/profile_t0/trh_1hz         dims={'height': 8, 'time': 3600}
```

Spot-check a few real values to be sure the data isn't all NaN:

In [ ]:
leaf = tree_test["/array/sonic_50hz"].ds
print(leaf.wind_u.isel(site=0, time=slice(0, 5)).compute())

## Stage 6: Restructure into an analysis-ready DataTree

Call `restructure_m2hats` on the full combined store in dry-run mode (`write=False`) to build the DataTree in memory. This reorganizes the flat variable-per-site layout into a hierarchy of instrument groups (e.g. `/array/sonic_60hz`, `/profile_t0/irga_60hz`) with proper site and height coordinates. The write step follows.

In [ ]:
tree = restructure_m2hats(ds, gdex_zarr, write=False, tilt_corrected=False)

### Write the restructured Zarr store

Reload `m2hats_to_zarr` to pick up any last-minute edits, stamp t1-swap metadata into the root group attributes, then write each leaf group to Zarr one at a time. Writing groups individually keeps each Dask graph small and avoids the memory pressure of scheduling the entire store at once. After all groups are written, metadata is consolidated so the store can be opened efficiently without scanning every group.

In [ ]:
importlib.reload(m)
from m2hats_to_zarr import restructure_m2hats  # refresh the direct import too

TIME_CHUNK = 3600  # seconds

# Stamp the swap into root attrs so the store is self-documenting.
tree.attrs["t1_csat_swap_utc"] = "2023-08-23T13:00:00"
tree.attrs["t1_csat_swap_note"] = (
    "Site t1 sonic was a CSAT3 (30 Hz, site t1p) before this timestamp "
    "and a CSAT3A (60 Hz, site t1) after. Pre-swap data lives in "
    "/array/sonic_30hz under site t1p; post-swap data lives in "
    "/array/sonic_60hz under site t1. Each column is NaN-padded "
    "outside its valid period."
)

# Write root group (attrs only) first to initialise the store.
tree.ds.to_zarr(gdex_zarr, mode="w", consolidated=False)

# Write each leaf group separately — one Dask graph at a time.
for path in tree.groups:
    node = tree[path]
    if not node.ds.data_vars:
        continue
    enc = m._encoding_for(node.ds, TIME_CHUNK)
    # Align Dask chunks with Zarr encoding: non-time dims are written whole
    ds_w = node.ds.chunk({d: -1 for d in node.ds.dims if d != "time"})
    print(f"writing  {path:35s} ({ds_w.sizes.get('time', 0):>9,} steps)", flush=True)
    ds_w.to_zarr(gdex_zarr, group=path, mode="a", consolidated=False, encoding=enc)

zarr_lib.consolidate_metadata(gdex_zarr)
print("done.")

## Stage 7: Verify the final output

Open the finished Zarr store as a DataTree and inspect the `sonic_60hz` group as a sanity check on dimensions, coordinates, and data variables.

In [ ]:
tree = xr.open_datatree(
    gdex_zarr,
    engine="zarr",
    consolidated=True,
    chunks={},
)

In [ ]:
sonic = tree["/array/sonic_60hz"].ds

### Quantitative QA: compare Zarr output against raw NetCDF

For a single time step and site, pull the same data from both the finished Zarr store and the original raw NetCDF file, then check that values agree to within floating-point tolerance (`< 1e-5`). Mismatches are flagged with `✗`. Adjust `CHECK_TIME`, `CHECK_SITE`, and `CHECK_GROUP` to probe different instruments or periods (see comments for the t1 pre/post-swap cases).

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECK_TIME  = np.datetime64('2023-09-21T09:12:02.500000000')
CHECK_SITE  = "t0p"
CHECK_GROUP = "/array/sonic_60hz"
ISFS_PATH = Path(isfs_highrate_path)

# ── 1. Zarr ───────────────────────────────────────────────────────────────────
zarr_slice = (
    tree[CHECK_GROUP].ds
    .sel(time=CHECK_TIME, site=CHECK_SITE)
    .compute()
)

# ── 2. Raw NetCDF ─────────────────────────────────────────────────────────────
ts        = pd.Timestamp(CHECK_TIME.astype("datetime64[ms]").item())
raw_file  = ISFS_PATH / f"isfs_m2hats_qc_geo_hr_{ts.strftime('%Y%m%d_%H')}.nc"
raw_ds    = fix_time_and_t1(xr.open_dataset(raw_file, decode_timedelta=False))
raw_slice = raw_ds.sel(time=CHECK_TIME, method="nearest")

# Sample dim for the group
sample_dim = {"60hz": "sample", "30hz": "sample_30", "50hz": "sample_50"}
sdim = next(v for k, v in sample_dim.items() if k in CHECK_GROUP)

raw_site = {
    v: raw_slice[v].values
    for v in raw_ds.data_vars
    if v.endswith(f"_{CHECK_SITE}") and sdim in raw_ds[v].dims
}

# ── 3. Compare ────────────────────────────────────────────────────────────────
rev_map = {meta[0]: short for short, meta in VAR_METADATA.items()}

print(f"site={CHECK_SITE}  time={CHECK_TIME}")
print(f"zarr group : {CHECK_GROUP}")
print(f"raw file   : {raw_file.name}\n")
print(f"{'zarr var':25s}  {'raw var':25s}  {'match':>6s}  {'max_diff':>12s}")
print("─" * 75)

for zarr_var in sorted(zarr_slice.data_vars):
    isfs_short = rev_map.get(zarr_var)
    if isfs_short is None:
        continue
    candidates = [v for v in raw_site if v.startswith(f"{isfs_short}_")]
    if not candidates:
        print(f"{zarr_var:25s}  {'(no match in raw)':25s}")
        continue
    raw_var = candidates[0]

    z = zarr_slice[zarr_var].values.astype("float64")
    r = raw_site[raw_var].astype("float64")

    both_nan = np.isnan(z) & np.isnan(r)
    match    = bool(np.all(both_nan | (np.abs(z - r) < 1e-5)))
    max_diff = float(np.nanmax(np.abs(z - r))) if not np.all(both_nan) else float("nan")

    print(f"{zarr_var:25s}  {raw_var:25s}  {'✓' if match else '✗':>6s}  {max_diff:>12.2e}")

raw_ds.close()

## Cleanup

Shut down the Dask cluster and client to release PBS resources.

In [ ]:
cluster.close()
client.close()